# 🗳️ ONPE – Consulta Electoral Masiva
Consulta automática de DNIs en https://consultaelectoral.onpe.gob.pe/inicio

**Pasos:**
1. Ejecuta la **Celda 1** (instalar dependencias) — solo la primera vez
2. Ejecuta la **Celda 2** (configuración)
3. Ejecuta la **Celda 3** para iniciar las consultas

In [ ]:
# ─── CELDA 1: Instalar dependencias ───────────────────────────────────────────
import subprocess, sys

# Instalar Chrome + ChromeDriver en Colab
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'chromium-browser', 'chromium-chromedriver'], check=True)

# Instalar paquetes Python
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'undetected-chromedriver>=3.5.0',
                'selenium>=4.15.0',
                'openpyxl>=3.1.0',
                'pandas'], check=True)

print('✅ Dependencias instaladas correctamente.')

In [ ]:
# ─── CELDA 2: Configuración ────────────────────────────────────────────────────

# ── Opción A: ingresar DNIs directamente aquí ──
DNIS_MANUALES = [
    # '12345678',
    # '87654321',
]

# ── Opción B: subir un archivo CSV/TXT/XLSX ──
# Descomenta las siguientes líneas para subir un archivo:
# from google.colab import files
# uploaded = files.upload()          # abre el selector de archivos
# ARCHIVO_DNIS = list(uploaded.keys())[0]
ARCHIVO_DNIS = None   # ← pon aquí la ruta si ya tienes el archivo subido

# ── Opciones de consulta ──
PAUSA_SEGUNDOS  = 10    # segundos entre consultas (respetar rate-limit ONPE)
SALTAR_YA_VISTOS = True  # True = no repetir DNIs ya consultados en esta sesión

print('✅ Configuración lista.')

In [ ]:
# ─── CELDA 3: Motor de consulta + ejecución ────────────────────────────────────
import re, time, random, sqlite3, csv, os
from datetime import datetime
from pathlib import Path
import pandas as pd
from IPython.display import display, clear_output

ONPE_URL = 'https://consultaelectoral.onpe.gob.pe/inicio'
DB_FILE  = '/content/onpe_consultas.db'

# ══════════════════════════════════════════════════════════════════════════════
# Base de datos
# ══════════════════════════════════════════════════════════════════════════════
class Database:
    def __init__(self, path=DB_FILE):
        self.path = path
        self._init()

    def _init(self):
        with sqlite3.connect(self.path) as c:
            c.execute('''
                CREATE TABLE IF NOT EXISTS consultas (
                    id           INTEGER PRIMARY KEY AUTOINCREMENT,
                    dni          TEXT NOT NULL UNIQUE,
                    nombres      TEXT,
                    region       TEXT,
                    provincia    TEXT,
                    distrito     TEXT,
                    miembro_mesa INTEGER DEFAULT 0,
                    local_vot    TEXT,
                    nro_mesa     TEXT,
                    nro_orden    TEXT,
                    estado       TEXT DEFAULT "pendiente",
                    error_msg    TEXT,
                    consultado   TEXT
                )
            ''')
            c.commit()

    def upsert(self, r):
        with sqlite3.connect(self.path) as c:
            c.execute('''
                INSERT OR REPLACE INTO consultas
                (dni, nombres, region, provincia, distrito, miembro_mesa,
                 local_vot, nro_mesa, nro_orden, estado, error_msg, consultado)
                VALUES (?,?,?,?,?,?,?,?,?,?,?,?)
            ''', (
                r.get('dni',''),      r.get('nombres',''),
                r.get('region',''),   r.get('provincia',''),
                r.get('distrito',''), 1 if r.get('miembro_mesa') else 0,
                r.get('local_vot',''),r.get('nro_mesa',''),
                r.get('nro_orden',''),r.get('estado','ok'),
                r.get('error_msg',''),datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            ))
            c.commit()

    def get_all(self):
        with sqlite3.connect(self.path) as c:
            c.row_factory = sqlite3.Row
            return [dict(r) for r in c.execute(
                'SELECT * FROM consultas ORDER BY id DESC').fetchall()]

    def done_dnis(self):
        with sqlite3.connect(self.path) as c:
            return {r[0] for r in c.execute(
                "SELECT dni FROM consultas WHERE estado='ok'").fetchall()}

    def export_csv(self, path):
        rows = self.get_all()
        if not rows: return False
        with open(path, 'w', newline='', encoding='utf-8-sig') as f:
            dw = csv.DictWriter(f, fieldnames=rows[0].keys())
            dw.writeheader()
            dw.writerows(rows)
        return True

# ══════════════════════════════════════════════════════════════════════════════
# Scraper (adaptado para Linux/Colab)
# ══════════════════════════════════════════════════════════════════════════════
class ONPEScraper:
    def __init__(self, log_fn=None):
        self.log     = log_fn or print
        self._driver = None

    def start(self):
        import undetected_chromedriver as uc
        self.log('Iniciando Chrome (modo headless para Colab)...')
        options = uc.ChromeOptions()
        options.add_argument('--headless=new')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--window-size=1366,768')
        options.add_argument('--lang=es-PE')
        options.binary_location = '/usr/bin/chromium-browser'
        self._driver = uc.Chrome(options=options)
        self._driver.get(ONPE_URL)
        time.sleep(random.uniform(4.0, 6.0))
        self._human_behavior()
        time.sleep(random.uniform(2.0, 3.0))
        self.log('Navegador listo.')

    def query_dni(self, dni):
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
        from selenium.webdriver.support import expected_conditions as EC
        from selenium.webdriver.common.keys import Keys

        dni = str(dni).strip().zfill(8)
        r   = self._empty(dni)
        try:
            wait = WebDriverWait(self._driver, 25)
            dni_input = None
            for css in ['input[type="tel"]', 'input[maxlength="8"]', 'input']:
                try:
                    dni_input = WebDriverWait(self._driver, 25).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, css)))
                    break
                except Exception:
                    continue

            if not dni_input:
                raise RuntimeError('No se encontró el campo DNI')

            self._driver.execute_script(
                'arguments[0].scrollIntoView({block:"center"}); arguments[0].click();',
                dni_input)
            time.sleep(random.uniform(0.2, 0.4))
            dni_input.send_keys(Keys.CONTROL + 'a')
            time.sleep(0.1)
            dni_input.send_keys(Keys.DELETE)
            time.sleep(0.2)
            for ch in dni:
                dni_input.send_keys(ch)
                time.sleep(random.uniform(0.08, 0.18))

            time.sleep(random.uniform(0.5, 1.0))
            btn = None
            for xpath in [
                "//button[contains(translate(text(),'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'CONSULTAR')]",
                "//button[@type='submit']",
                '//button']:
                try:
                    btn = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
                    break
                except Exception:
                    continue

            if not btn:
                raise RuntimeError('No se encontró el botón CONSULTAR')

            self._driver.execute_script(
                'arguments[0].scrollIntoView({block:"center"});', btn)
            time.sleep(0.3)
            self._driver.execute_script('arguments[0].click();', btn)

            def _ready(d):
                txt = (d.execute_script('return document.body.textContent;') or '').lower()
                return ('no eres miembro de mesa' in txt
                        or 'sí eres miembro de mesa' in txt
                        or 'nombres y apellidos' in txt
                        or 'error interno del servidor' in txt)
            try:
                WebDriverWait(self._driver, 15).until(_ready)
            except Exception:
                pass

            r = self._extract(dni)
        except Exception as e:
            r['estado']    = 'error'
            r['error_msg'] = str(e)
        return r

    def _extract(self, dni):
        from selenium.webdriver.common.by import By
        r = self._empty(dni)
        try:
            body = self._driver.execute_script('return document.body.textContent;') or ''
            body_up = body.upper()

            if 'ERROR INTERNO' in body_up or 'VOLVER AL INICIO' in body_up:
                r['estado'] = 'error'
                r['error_msg'] = 'Error interno del servidor ONPE'
                return r

            def _css(sel):
                try:
                    el = self._driver.find_element(By.CSS_SELECTOR, sel)
                    return (el.get_attribute('textContent') or el.text or '').strip()
                except Exception:
                    return ''

            r['miembro_mesa'] = ('NO ERES MIEMBRO DE MESA' not in body_up
                                 and ('SÍ ERES MIEMBRO DE MESA' in body_up
                                      or 'SI ERES MIEMBRO DE MESA' in body_up))

            nombre = _css('.apellido') or _css('.nombre-completo')
            if nombre and not nombre.isdigit() and len(nombre) > 3:
                r['nombres'] = nombre

            geo = _css('.local')
            if '/' in geo:
                parts = geo.split('/')
                r['region']    = parts[0].strip() if len(parts) > 0 else ''
                r['provincia'] = parts[1].strip() if len(parts) > 1 else ''
                r['distrito']  = parts[2].strip() if len(parts) > 2 else ''

            lines = [l.strip() for l in body.splitlines() if l.strip()]

            if not r['nombres']:
                for i, line in enumerate(lines):
                    if re.search(r'nombres?\s*y\s*apellidos?', line, re.I):
                        for j in range(i+1, min(i+4, len(lines))):
                            if len(lines[j]) > 5 and not lines[j].isdigit():
                                r['nombres'] = lines[j]
                                break
                        break

            if not r['region']:
                for line in lines:
                    parts = line.split('/')
                    if len(parts) == 3 and all(len(p.strip()) > 2 for p in parts):
                        if not re.search(r'\d', line):
                            r['region']    = parts[0].strip()
                            r['provincia'] = parts[1].strip()
                            r['distrito']  = parts[2].strip()
                            break

            lv = _css('.local-votacion') or _css('app-caso-1 .nombre-local')
            if lv:
                r['local_vot'] = lv

            mv = _css('.nro-mesa')
            if mv and re.search(r'\d', mv):
                r['nro_mesa'] = re.findall(r'\d+', mv)[0]

            ov = _css('.nro-orden')
            if ov and re.search(r'\d', ov):
                r['nro_orden'] = re.findall(r'\d+', ov)[0]

            r['estado'] = 'ok'
        except Exception as e:
            r['estado']    = 'error'
            r['error_msg'] = str(e)
        return r

    def back_to_form(self):
        from selenium.webdriver.common.by import By
        from selenium.webdriver.support.ui import WebDriverWait
        from selenium.webdriver.support import expected_conditions as EC
        try:
            for xpath in ["//a[contains(text(),'Salir')]",
                          "//button[contains(text(),'Salir')]"]:
                try:
                    el = WebDriverWait(self._driver, 3).until(
                        EC.element_to_be_clickable((By.XPATH, xpath)))
                    self._driver.execute_script('arguments[0].click();', el)
                    time.sleep(random.uniform(2.0, 3.0))
                    return
                except Exception:
                    continue
        except Exception:
            pass
        self._driver.get(ONPE_URL)
        time.sleep(random.uniform(2.5, 4.0))
        self._human_behavior()

    def _human_behavior(self):
        from selenium.webdriver.common.action_chains import ActionChains
        from selenium.webdriver.common.by import By
        try:
            body = self._driver.find_element(By.TAG_NAME, 'body')
            ActionChains(self._driver).move_to_element(body).perform()
            time.sleep(random.uniform(0.3, 0.5))
            for _ in range(random.randint(3, 5)):
                try:
                    ActionChains(self._driver).move_to_element_with_offset(
                        body,
                        random.randint(-150, 150),
                        random.randint(-80, 80)).perform()
                except Exception:
                    pass
                time.sleep(random.uniform(0.1, 0.3))
        except Exception:
            pass

    def _empty(self, dni):
        return {'dni': dni, 'nombres': '', 'region': '', 'provincia': '',
                'distrito': '', 'miembro_mesa': False, 'local_vot': '',
                'nro_mesa': '', 'nro_orden': '', 'estado': 'pendiente', 'error_msg': ''}

    def stop(self):
        try:
            if self._driver: self._driver.quit()
        except Exception:
            pass

# ══════════════════════════════════════════════════════════════════════════════
# Cargar DNIs (archivo o lista manual)
# ══════════════════════════════════════════════════════════════════════════════
def cargar_dnis_archivo(path):
    dnis = []
    ext = Path(path).suffix.lower()
    if ext in ('.xlsx', '.xls'):
        import openpyxl
        wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
        for ws in wb.worksheets:
            for row in ws.iter_rows(values_only=True):
                for cell in row:
                    if cell is None: continue
                    s = str(int(cell)) if isinstance(cell, (int, float)) else str(cell).strip()
                    m = re.search(r'\b(\d{7,9})\b', s)
                    if m: dnis.append(m.group(1).zfill(8))
    else:
        with open(path, 'r', encoding='utf-8-sig', errors='ignore') as f:
            for line in f:
                for m in re.finditer(r'\b(\d{7,9})\b', line):
                    dnis.append(m.group(1).zfill(8))
    seen, unique = set(), []
    for d in dnis:
        if d not in seen:
            seen.add(d)
            unique.append(d)
    return unique

# ══════════════════════════════════════════════════════════════════════════════
# EJECUCIÓN PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════
# Construir lista de DNIs
dnis = list(DNIS_MANUALES)
if ARCHIVO_DNIS:
    dnis += cargar_dnis_archivo(ARCHIVO_DNIS)

# Deduplicar
seen, dnis = set(), [d for d in dnis if d not in seen and not seen.add(d)]

if not dnis:
    print('⚠️  No hay DNIs para consultar. Agrega DNIs en DNIS_MANUALES o sube un archivo.')
else:
    db = Database()
    logs = []

    if SALTAR_YA_VISTOS:
        ya = db.done_dnis()
        antes = len(dnis)
        dnis = [d for d in dnis if d not in ya]
        if antes != len(dnis):
            print(f'ℹ️  {antes - len(dnis)} DNIs ya consultados, se omiten.')

    if not dnis:
        print('✅ Todos los DNIs ya están consultados.')
    else:
        print(f'🔍 Iniciando {len(dnis)} consultas (pausa {PAUSA_SEGUNDOS}s entre cada una)...')
        print('─' * 60)

        def log(msg):
            ts = datetime.now().strftime('%H:%M:%S')
            entry = f'[{ts}] {msg}'
            logs.append(entry)
            print(entry)

        scraper = ONPEScraper(log_fn=log)
        resultados = []

        try:
            scraper.start()
            total = len(dnis)

            for i, dni in enumerate(dnis):
                log(f'[{i+1}/{total}] Consultando DNI: {dni}')
                r = scraper.query_dni(dni)
                db.upsert(r)
                resultados.append(r)

                miembro = '✅ MIEMBRO MESA' if r.get('miembro_mesa') else '❌ no miembro'
                log(f'  → {r["estado"].upper()} | {miembro} | {r.get("nombres", "")}')

                if i < total - 1:
                    scraper.back_to_form()
                    log(f'  ⏳ Esperando {PAUSA_SEGUNDOS}s...')
                    time.sleep(PAUSA_SEGUNDOS)
        finally:
            scraper.stop()

        # ── Mostrar resultados como tabla ──
        print('\n' + '═' * 60)
        print('RESULTADOS FINALES')
        print('═' * 60)
        df = pd.DataFrame(resultados)
        df['miembro_mesa'] = df['miembro_mesa'].map({True: 'SÍ ✓', False: 'NO'})
        cols_show = ['dni','nombres','region','provincia','distrito',
                     'miembro_mesa','local_vot','nro_mesa','nro_orden','estado']
        display(df[[c for c in cols_show if c in df.columns]])

        total_ok  = sum(1 for r in resultados if r.get('estado') == 'ok')
        miembros  = sum(1 for r in resultados if r.get('miembro_mesa'))
        errores   = sum(1 for r in resultados if r.get('estado') == 'error')
        print(f'\nTotal: {total_ok+errores}  |  Miembros de mesa: {miembros}  |  Errores: {errores}')

        print('\n✅ Proceso completado.')

In [ ]:
# ─── CELDA 4 (opcional): Exportar resultados a CSV y descargar ─────────────────
from google.colab import files

export_path = f'/content/onpe_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
db = Database()
if db.export_csv(export_path):
    print(f'📥 Descargando {export_path}...')
    files.download(export_path)
else:
    print('⚠️  No hay datos para exportar.')